# Populating DWH — SCD Type 2

## Import libraries

In [485]:
import sqlite3
import pandas as pd
from datetime import date

VANDAAG      = date.today().isoformat()   # bijv. '2025-05-02'
VER_VERLEDEN = '1900-01-01'
EINDE_TIJD   = '9999-12-31'

## Define connections

In [486]:
SDM = sqlite3.connect('../SDM/BikeToDriveSDM.db')
DWH = sqlite3.connect('BikeToDriveDWH_SCD2.db')

In [ ]:
def _leeg_tabel(tabel_naam, dwh_conn):
    dwh_conn.execute(f"DELETE FROM {tabel_naam}")
    dwh_conn.commit()


def laad_scd2(df_nieuw, tabel_naam, business_key, check_kolommen, dwh_conn,
              eerste_run=False):
    df_nieuw = df_nieuw.copy()

    if eerste_run:
        # Leeg de tabel (behoudt structuur + foreign keys) en insert opnieuw
        _leeg_tabel(tabel_naam, dwh_conn)
        df_nieuw['geldig_van'] = VER_VERLEDEN
        df_nieuw['geldig_tot'] = EINDE_TIJD
        df_nieuw['is_huidig']  = 1
        df_nieuw.to_sql(tabel_naam, dwh_conn, if_exists='append', index=False)
        print(f"[{tabel_naam}] Initiële load: {len(df_nieuw)} rijen geladen.")
        return df_nieuw

    # --- Volgende runs: detecteer wijzigingen ---
    df_dwh = pd.read_sql_query(
        f"SELECT * FROM {tabel_naam} WHERE is_huidig = 1", dwh_conn
    )

    df_merge = df_nieuw.merge(
        df_dwh[[business_key] + check_kolommen],
        on=business_key, suffixes=('_nieuw', '_dwh'), how='left'
    )

    masker = pd.Series([False] * len(df_merge), index=df_merge.index)
    for kol in check_kolommen:
        masker = masker | (df_merge[f'{kol}_nieuw'].astype(str) !=
                           df_merge[f'{kol}_dwh'].astype(str))

    gewijzigde_keys = df_merge.loc[masker, business_key].tolist()

    # Nieuwe entiteiten (nog niet in DWH)
    bestaande_keys  = df_dwh[business_key].tolist()
    nieuwe_keys     = df_nieuw.loc[
        ~df_nieuw[business_key].isin(bestaande_keys), business_key
    ].tolist()

    cursor = dwh_conn.cursor()

    if gewijzigde_keys:
        placeholders = ','.join('?' * len(gewijzigde_keys))
        cursor.execute(
            f"""UPDATE {tabel_naam}
               SET geldig_tot = date(?, '-1 day'),
                   is_huidig  = 0
             WHERE {business_key} IN ({placeholders})
               AND is_huidig = 1""",
            [VANDAAG] + gewijzigde_keys
        )

    te_inserteren_keys = gewijzigde_keys + nieuwe_keys
    if te_inserteren_keys:
        df_insert = df_nieuw[
            df_nieuw[business_key].isin(te_inserteren_keys)
        ].copy()
        df_insert['geldig_van'] = VANDAAG
        df_insert['geldig_tot'] = EINDE_TIJD
        df_insert['is_huidig']  = 1
        df_insert.to_sql(tabel_naam, dwh_conn, if_exists='append', index=False)

    dwh_conn.commit()
    print(f"[{tabel_naam}] Gewijzigd: {len(gewijzigde_keys)} | "
          f"Nieuw: {len(nieuwe_keys)} | Ongewijzigd: "
          f"{len(df_nieuw) - len(gewijzigde_keys) - len(nieuwe_keys)}")

    return pd.read_sql_query(f"SELECT * FROM {tabel_naam}", dwh_conn)


def haal_sk_op(df_feit, dim_df, business_key, datum_kolom, sk_kolom):
    """
    Koppel de juiste surrogate key aan een feitentabel via een temporele join.
    Zoekt de dimensierij die geldig was op de transactiedatum.

    Parameters
    ----------
    df_feit      : feitentabel DataFrame
    dim_df       : dimensietabel DataFrame (moet geldig_van, geldig_tot bevatten)
    business_key : kolom waarop gekoppeld wordt (bijv. 'klantnr')
    datum_kolom  : datumkolom in de feitentabel (bijv. 'datum')
    sk_kolom     : naam van de surrogate-key kolom in de dim (bijv. 'klant_sk')
    """
    dim_subset = dim_df[[business_key, 'geldig_van', 'geldig_tot', sk_kolom]].copy()
    df = df_feit.merge(dim_subset, on=business_key, how='left')
    df = df[
        (df[datum_kolom] >= df['geldig_van']) &
        (df[datum_kolom] <= df['geldig_tot'])
    ].drop(columns=['geldig_van', 'geldig_tot'])
    return df

## Dim_Klant

In [488]:
df_klant_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_klant', SDM)
df_klant_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_klant', SDM)

df_klant_av = df_klant_av.drop('klantnr', axis=1)
df_klant_fv = df_klant_fv.drop('klantnr', axis=1)

df_klant_combined = pd.concat([df_klant_av, df_klant_fv]).drop_duplicates().reset_index(drop=True)
df_klant_combined.insert(0, 'klantnr', range(1, len(df_klant_combined) + 1))

# Voeg surrogate key toe (rowid volstaat niet; we maken een expliciete sk)
df_klant_combined.insert(0, 'klant_sk', range(1, len(df_klant_combined) + 1))

df_klant_scd2 = laad_scd2(
    df_nieuw       = df_klant_combined,
    tabel_naam     = 'Dim_Klant',
    business_key   = 'klantnr',
    check_kolommen = ['adres', 'woonplaats'],
    dwh_conn       = DWH,
    eerste_run     = True      # ← zet op False bij volgende runs
)

df_klant_scd2

[Dim_Klant] Initiële load: 30 rijen geladen.


,klant_sk,klantnr,naam,adres,woonplaats,geslacht,geboortedatum,geldig_van,geldig_tot,is_huidig
0,1,1,Jan Jansen,Kerkstraat 12,Amsterdam,M,1985-03-22,1900-01-01,9999-12-31,1
1,2,2,Sophie de Boer,Lindelaan 8,Rotterdam,V,1990-07-11,1900-01-01,9999-12-31,1
2,3,3,Pieter Visser,Havenstraat 3,Den Haag,M,1978-11-05,1900-01-01,9999-12-31,1
3,4,4,Emma Smit,Boomgaard 22,Haarlem,V,1995-02-18,1900-01-01,9999-12-31,1
4,5,5,Tom Bakker,Stationsweg 44,Leiden,M,1982-09-09,1900-01-01,9999-12-31,1
5,6,6,Lisa Meijer,Dijkstraat 10,Zaandam,V,1993-12-30,1900-01-01,9999-12-31,1
6,7,7,Bart de Vries,Brouwersgracht 7,Delft,M,1976-06-06,1900-01-01,9999-12-31,1
7,8,8,Julia van Dijk,Plataanlaan 5,Hoorn,V,2000-01-15,1900-01-01,9999-12-31,1
8,9,9,Kevin Mol,Singel 99,Alkmaar,M,1989-08-01,1900-01-01,9999-12-31,1
9,10,10,Nina Groen,Waterstraat 16,Schiedam,V,1991-05-25,1900-01-01,9999-12-31,1


## Dim_Filiaal

In [489]:
df_filiaal_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_filiaal', SDM)
df_filiaal_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_filiaal', SDM)
df_filiaal_o  = pd.read_sql_query('SELECT * FROM onderhoud_filiaal', SDM)

df_filiaal_av = df_filiaal_av.drop('filiaalnr', axis=1)
df_filiaal_fv = df_filiaal_fv.drop('filiaalnr', axis=1)
df_filiaal_o  = df_filiaal_o.drop('filiaalnr', axis=1)

df_filiaal_combined = pd.concat([df_filiaal_av, df_filiaal_fv, df_filiaal_o]).drop_duplicates().reset_index(drop=True)
df_filiaal_combined.insert(0, 'filiaalnr', range(1, len(df_filiaal_combined) + 1))
df_filiaal_combined.insert(0, 'filiaal_sk', range(1, len(df_filiaal_combined) + 1))

df_filiaal_scd2 = laad_scd2(
    df_nieuw       = df_filiaal_combined,
    tabel_naam     = 'Dim_Filiaal',
    business_key   = 'filiaalnr',
    check_kolommen = ['adres', 'provincie'],
    dwh_conn       = DWH,
    eerste_run     = True
)

df_filiaal_scd2

[Dim_Filiaal] Initiële load: 5 rijen geladen.


,filiaal_sk,filiaalnr,naam,adres,provincie,geldig_van,geldig_tot,is_huidig
0,1,1,BikeWorld Amsterdam,Prinsengracht 100,Noord-Holland,1900-01-01,9999-12-31,1
1,2,2,FietsGigant Haarlem,Grote Markt 22,Noord-Holland,1900-01-01,9999-12-31,1
2,3,3,FietsExpress Rotterdam,Coolsingel 55,Zuid-Holland,1900-01-01,9999-12-31,1
3,4,4,CycleCity Leiden,Breestraat 48,Zuid-Holland,1900-01-01,9999-12-31,1
4,5,5,ZaanFiets Zaandam,Damstraat 5,Noord-Holland,1900-01-01,9999-12-31,1


## Dim_Monteur

In [490]:
df_monteur_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_monteur', SDM)
df_monteur_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_monteur', SDM)
df_monteur_o  = pd.read_sql_query('SELECT * FROM onderhoud_monteur', SDM)

df_monteur_av = df_monteur_av.drop('monteurnr', axis=1)
df_monteur_fv = df_monteur_fv.drop('monteurNr', axis=1)
df_monteur_o  = df_monteur_o.drop('monteurNr', axis=1)

df_monteur_combined = pd.concat([df_monteur_av, df_monteur_fv, df_monteur_o]).drop_duplicates().reset_index(drop=True)
df_monteur_combined.insert(0, 'monteurnr', range(1, len(df_monteur_combined) + 1))
df_monteur_combined.insert(0, 'monteur_sk', range(1, len(df_monteur_combined) + 1))

df_monteur_scd2 = laad_scd2(
    df_nieuw       = df_monteur_combined,
    tabel_naam     = 'Dim_Monteur',
    business_key   = 'monteurnr',
    check_kolommen = ['woonplaats', 'uurloon', 'filiaal'],
    dwh_conn       = DWH,
    eerste_run     = True
)

df_monteur_scd2

[Dim_Monteur] Initiële load: 16 rijen geladen.


,monteur_sk,monteurnr,naam,woonplaats,uurloon,filiaal,geldig_van,geldig_tot,is_huidig
0,1,1,Tom van Dijk,Amsterdam,19.50,1,1900-01-01,9999-12-31,1
1,2,2,Lisa Verhoeven,Amstelveen,18.75,1,1900-01-01,9999-12-31,1
2,3,3,Kevin Jaspers,Zaandam,20.00,1,1900-01-01,9999-12-31,1
3,4,4,Romy Willems,Haarlem,17.80,2,1900-01-01,9999-12-31,1
4,5,5,Daan de Groot,Rotterdam,21.00,3,1900-01-01,9999-12-31,1
5,6,6,Sophie de Leeuw,Schiedam,19.90,3,1900-01-01,9999-12-31,1
6,7,7,Mark van Vliet,Dordrecht,20.25,3,1900-01-01,9999-12-31,1
7,8,8,Jesse Peters,Leiden,16.95,4,1900-01-01,9999-12-31,1
8,9,9,Eva Meijer,Zoetermeer,18.40,4,1900-01-01,9999-12-31,1
9,10,10,Milan de Bruin,Zaandam,17.25,2,1900-01-01,9999-12-31,1


## Dim_Partner


In [491]:
df_leverancier_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_leverancier', SDM)
df_leverancier_ai = pd.read_sql_query('SELECT * FROM accessoire_inkoop_leverancier', SDM)
df_fabrikant_o    = pd.read_sql_query('SELECT * FROM onderhoud_fabrikant', SDM)
df_fabrikant_fv   = pd.read_sql_query('SELECT * FROM fiets_verkoop_fabrikant', SDM)
df_fabrikant_fi   = pd.read_sql_query('SELECT * FROM fiets_inkoop_fabrikant', SDM)

df_leverancier_av = df_leverancier_av.drop('leveranciernr', axis=1)
df_leverancier_ai = df_leverancier_ai.drop('leveranciernr', axis=1)
df_fabrikant_o    = df_fabrikant_o.drop('fabrikantnr', axis=1)
df_fabrikant_fv   = df_fabrikant_fv.drop('fabrikantnr', axis=1)
df_fabrikant_fi   = df_fabrikant_fi.drop('fabrikantnr', axis=1)

df_leverancier_combined = pd.concat([df_leverancier_av, df_leverancier_ai]).drop_duplicates().reset_index(drop=True)
df_fabrikant_combined   = pd.concat([df_fabrikant_o, df_fabrikant_fv, df_fabrikant_fi]).drop_duplicates().reset_index(drop=True)

df_leverancier_combined['type_partner'] = 'leverancier'
df_leverancier_combined.rename(columns={'woonplaats': 'plaats'}, inplace=True)
df_fabrikant_combined['type_partner'] = 'fabrikant'

df_partner_combined = pd.concat([df_leverancier_combined, df_fabrikant_combined]).drop_duplicates().reset_index(drop=True)
df_partner_combined.insert(0, 'partnernr', range(1, len(df_partner_combined) + 1))
df_partner_combined.insert(0, 'partner_sk', range(1, len(df_partner_combined) + 1))

df_partner_scd2 = laad_scd2(
    df_nieuw       = df_partner_combined,
    tabel_naam     = 'Dim_Partner',
    business_key   = 'partnernr',
    check_kolommen = ['adres', 'plaats'],
    dwh_conn       = DWH,
    eerste_run     = True
)

df_partner_scd2

[Dim_Partner] Initiële load: 16 rijen geladen.


,partner_sk,partnernr,naam,adres,plaats,type_partner,geldig_van,geldig_tot,is_huidig
0,1,1,BikeParts BV,Fietsenstraat 12,Amsterdam,leverancier,1900-01-01,9999-12-31,1
1,2,2,CycleSupplies,Rembrandtlaan 7,Rotterdam,leverancier,1900-01-01,9999-12-31,1
2,3,3,TwoWheels Trading,Marktplein 5,Den Haag,leverancier,1900-01-01,9999-12-31,1
3,4,4,DutchBike Import,Kade 22,Leiden,leverancier,1900-01-01,9999-12-31,1
4,5,5,UrbanGear NL,Stationsweg 15,Utrecht,leverancier,1900-01-01,9999-12-31,1
5,6,6,SpeedCycle BV,Spoorstraat 12,Eindhoven,fabrikant,1900-01-01,9999-12-31,1
6,7,7,UrbanBike Co.,Keizersgracht 45,Amsterdam,fabrikant,1900-01-01,9999-12-31,1
7,8,8,HollandRider,Marktstraat 9,Utrecht,fabrikant,1900-01-01,9999-12-31,1
8,9,9,EcoWheelers,Kade 33,Delft,fabrikant,1900-01-01,9999-12-31,1
9,10,10,DutchMotion,Westplein 7,Rotterdam,fabrikant,1900-01-01,9999-12-31,1


## Dim_Product

In [492]:
df_accessoire_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_accessoire', SDM)
df_accessoire_ai = pd.read_sql_query('SELECT * FROM accessoire_inkoop_accessoire', SDM)
df_fiets_o  = pd.read_sql_query('SELECT * FROM onderhoud_fiets', SDM)
df_fiets_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_fiets', SDM)
df_fiets_fi = pd.read_sql_query('SELECT * FROM fiets_inkoop_fiets', SDM)

df_leverancier_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_leverancier', SDM)
df_leverancier_ai = pd.read_sql_query('SELECT * FROM accessoire_inkoop_leverancier', SDM)
df_fabrikant_o  = pd.read_sql_query('SELECT * FROM onderhoud_fabrikant', SDM)
df_fabrikant_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_fabrikant', SDM)
df_fabrikant_fi = pd.read_sql_query('SELECT * FROM fiets_inkoop_fabrikant', SDM)

df_accessoire_av = pd.merge(df_accessoire_av, df_leverancier_av, left_on='leverancier', right_on='leveranciernr', how='left', suffixes=('_accessoire', '_leverancier'))
df_accessoire_av.rename(columns={'naam_accessoire': 'type_product'}, inplace=True)
df_accessoire_av.drop(columns=['leveranciernr', 'leverancier', 'adres', 'woonplaats'], inplace=True, errors='ignore')
df_accessoire_av = pd.merge(df_accessoire_av, df_partner_combined, left_on='naam_leverancier', right_on='naam', how='left')
df_accessoire_av.drop(columns=['naam_leverancier', 'naam', 'adres', 'plaats', 'type_partner'], inplace=True, errors='ignore')

df_accessoire_ai = pd.merge(df_accessoire_ai, df_leverancier_ai, left_on='leverancier', right_on='leveranciernr', how='left', suffixes=('_accessoire', '_leverancier'))
df_accessoire_ai.rename(columns={'naam_accessoire': 'type_product'}, inplace=True)
df_accessoire_ai.drop(columns=['leveranciernr', 'leverancier', 'adres', 'woonplaats'], inplace=True, errors='ignore')
df_accessoire_ai = pd.merge(df_accessoire_ai, df_partner_combined, left_on='naam_leverancier', right_on='naam', how='left')
df_accessoire_ai.drop(columns=['naam_leverancier', 'naam', 'adres', 'plaats', 'type_partner'], inplace=True, errors='ignore')

df_fiets_o = pd.merge(df_fiets_o, df_fabrikant_o, left_on='fabrikant', right_on='fabrikantnr', how='left')
df_fiets_o.drop(columns=['fabrikantnr', 'fabrikant', 'adres', 'woonplaats', 'plaats'], inplace=True, errors='ignore')
df_fiets_o = pd.merge(df_fiets_o, df_partner_combined, left_on='naam', right_on='naam', how='left')
df_fiets_o.drop(columns=['naam_fabrikant', 'naam', 'adres', 'plaats', 'type_partner'], inplace=True, errors='ignore')
df_fiets_o.rename(columns={'type': 'type_product'}, inplace=True)

df_fiets_fv = pd.merge(df_fiets_fv, df_fabrikant_fv, left_on='fabrikant', right_on='fabrikantnr', how='left')
df_fiets_fv.drop(columns=['fabrikantnr', 'fabrikant', 'adres', 'woonplaats', 'plaats'], inplace=True, errors='ignore')
df_fiets_fv = pd.merge(df_fiets_fv, df_partner_combined, left_on='naam', right_on='naam', how='left')
df_fiets_fv.drop(columns=['naam_fabrikant', 'naam', 'adres', 'plaats', 'type_partner'], inplace=True, errors='ignore')
df_fiets_fv.rename(columns={'type': 'type_product'}, inplace=True)

df_fiets_fi = pd.merge(df_fiets_fi, df_fabrikant_fi, left_on='fabrikant', right_on='fabrikantnr', how='left')
df_fiets_fi.drop(columns=['fabrikantnr', 'fabrikant', 'adres', 'woonplaats', 'plaats'], inplace=True, errors='ignore')
df_fiets_fi = pd.merge(df_fiets_fi, df_partner_combined, left_on='naam', right_on='naam', how='left')
df_fiets_fi.drop(columns=['naam_fabrikant', 'naam', 'adres', 'plaats', 'type_partner'], inplace=True, errors='ignore')
df_fiets_fi.rename(columns={'type': 'type_product'}, inplace=True)

df_accessoire_av = df_accessoire_av.drop('accessoirenr', axis=1)
df_accessoire_ai = df_accessoire_ai.drop('accessoirenr', axis=1)
df_fiets_o  = df_fiets_o.drop('fietsnr', axis=1)
df_fiets_fv = df_fiets_fv.drop('fietsnr', axis=1)
df_fiets_fi = df_fiets_fi.drop('fietsnr', axis=1)

df_accessoire_combined = pd.concat([df_accessoire_av, df_accessoire_ai]).drop_duplicates().reset_index(drop=True)
df_accessoire_combined['categorie'] = 'accessoire'
df_fiets_combined = pd.concat([df_fiets_o, df_fiets_fv, df_fiets_fi]).drop_duplicates().reset_index(drop=True)
df_fiets_combined['categorie'] = 'fiets'
df_product_combined = pd.concat([df_accessoire_combined, df_fiets_combined]).drop_duplicates().reset_index(drop=True)

df_product_combined['omzet'] = df_product_combined['standaardprijs'] - df_product_combined['inkoopprijs']
df_product_combined.insert(0, 'productnr', range(1, len(df_product_combined) + 1))
df_product_combined.insert(0, 'product_sk', range(1, len(df_product_combined) + 1))

df_product_scd2 = df_product_combined.copy()
df_product_scd2['geldig_van'] = VER_VERLEDEN
df_product_scd2['geldig_tot'] = EINDE_TIJD
df_product_scd2['is_huidig'] = 1

_leeg_tabel('Dim_Product', DWH)
df_product_scd2.to_sql('Dim_Product', DWH, if_exists='append', index=False)
df_product_scd2

df_product_scd2

,product_sk,productnr,type_product,standaardprijs,inkoopprijs,soort,partner_sk,partnernr,categorie,merk,kleur,omzet,geldig_van,geldig_tot,is_huidig
0,1,1,LED voorlamp,14.99,8.50,Verlichting,1,1,accessoire,NaN,NaN,6.49,1900-01-01,9999-12-31,1
1,2,2,LED achterlicht,12.99,7.25,Verlichting,1,1,accessoire,NaN,NaN,5.74,1900-01-01,9999-12-31,1
2,3,3,USB-oplaadbare fietslamp,24.95,15.00,Verlichting,1,1,accessoire,NaN,NaN,9.95,1900-01-01,9999-12-31,1
3,4,4,Ringslot AXA,19.95,11.40,Sloten,1,1,accessoire,NaN,NaN,8.55,1900-01-01,9999-12-31,1
4,5,5,Ketting met cijferslot,17.50,10.00,Sloten,2,2,accessoire,NaN,NaN,7.50,1900-01-01,9999-12-31,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,114,114,ELE-120,731.83,576.62,Elektrische fiets,14,14,fiets,FastFrame,Grijs,155.21,1900-01-01,9999-12-31,1
114,115,115,RAC-797,399.31,327.02,Racefiets,14,14,fiets,FastFrame,Rood,72.29,1900-01-01,9999-12-31,1
115,116,116,STA-974,731.99,622.99,Stadsfiets,15,15,fiets,OranjeFiets,Geel,109.00,1900-01-01,9999-12-31,1
116,117,117,RAC-701,589.50,513.51,Racefiets,15,15,fiets,OranjeFiets,Geel,75.99,1900-01-01,9999-12-31,1


## Dim_Datum & Dim_Maand

In [493]:
maanden_nl = [
    'Januari','Februari','Maart','April','Mei','Juni',
    'Juli','Augustus','September','Oktober','November','December'
]
kwartalen = ['Q1','Q1','Q1','Q2','Q2','Q2','Q3','Q3','Q3','Q4','Q4','Q4']

df_maand = pd.DataFrame({
    'maand_id' : range(1, 13),
    'maandnr'  : range(1, 13),
    'jaar'     : [2024] * 12,
    'maand'    : maanden_nl,
    'kwartaal' : kwartalen,
})
_leeg_tabel('Dim_Maand', DWH)
df_maand.to_sql('Dim_Maand', DWH, if_exists='append', index=False)

maand_lookup = {(r.maandnr, r.jaar): r.maand_id for _, r in df_maand.iterrows()}

# Verzamel alle datums uit de brondatabases
datum_queries = [
    'SELECT datum FROM accessoire_verkoop_verkoop',
    'SELECT datum FROM fiets_verkoop_verkoop',
    'SELECT datum FROM onderhoud_onderhoud',
    'SELECT datum FROM accessoire_inkoop_inkoop',
    'SELECT datum FROM fiets_inkoop_inkoop',
]
all_dates = pd.Series(dtype='object')
for q in datum_queries:
    try:
        df_tmp = pd.read_sql_query(q, SDM)
        all_dates = pd.concat([all_dates, df_tmp.iloc[:, 0]])
    except Exception:
        pass

all_dates = pd.to_datetime(all_dates.dropna().unique())

df_datum = pd.DataFrame({'datum': all_dates})
df_datum['dag']      = df_datum['datum'].dt.day
df_datum['maandnr']  = df_datum['datum'].dt.month
df_datum['jaar']     = df_datum['datum'].dt.year
df_datum['maand_id'] = df_datum.apply(lambda r: maand_lookup.get((r['maandnr'], r['jaar'])), axis=1)
df_datum['datum']    = df_datum['datum'].dt.strftime('%Y-%m-%d')
df_datum = df_datum[['datum', 'dag', 'maand_id']]

_leeg_tabel('Dim_Datum', DWH)
df_datum.to_sql('Dim_Datum', DWH, if_exists='append', index=False)
df_datum

,datum,dag,maand_id
0,2024-01-23,23,1
1,2024-10-03,3,10
2,2024-10-30,30,10
3,2024-11-16,16,11
4,2024-01-02,2,1
...,...,...,...
191,2024-12-13,13,12
192,2024-08-10,10,8
193,2024-03-04,4,3
194,2024-07-03,3,7


## Feit_Verkoop

In [494]:
df_verkoop_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_verkoop', SDM)
df_verkoop_fv = pd.read_sql_query('SELECT * FROM fiets_verkoop_verkoop', SDM)

df_accessoire_av = pd.read_sql_query('SELECT * FROM accessoire_verkoop_accessoire', SDM)
df_monteur_av    = pd.read_sql_query('SELECT * FROM accessoire_verkoop_monteur', SDM)
df_klant_av      = pd.read_sql_query('SELECT * FROM accessoire_verkoop_klant', SDM)
df_fiets_fv      = pd.read_sql_query('SELECT * FROM fiets_verkoop_fiets', SDM)
df_monteur_fv    = pd.read_sql_query('SELECT * FROM fiets_verkoop_monteur', SDM)
df_klant_fv      = pd.read_sql_query('SELECT * FROM fiets_verkoop_klant', SDM)

# --- Accessoire-verkopen ---
df_verkoop_av = pd.merge(df_verkoop_av, df_accessoire_av, left_on='accessoire', right_on='accessoirenr', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['inkoopprijs', 'soort', 'leverancier', 'accessoirenr', 'accessoire'], errors='ignore')
df_verkoop_av = pd.merge(df_verkoop_av, df_product_combined[['type_product','productnr']], left_on='naam', right_on='type_product', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['naam', 'soort', 'standaardprijs_y', 'type'], errors='ignore')
df_verkoop_av.rename(columns={'standaardprijs_x': 'standaardprijs'}, inplace=True)

df_verkoop_av = pd.merge(df_verkoop_av, df_klant_fv, left_on='klant', right_on='klantnr', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['klantnr', 'geslacht', 'adres', 'woonplaats', 'geboortedatum'], errors='ignore')
df_verkoop_av = pd.merge(df_verkoop_av, df_klant_combined[['naam','klantnr']], left_on='naam', right_on='naam', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['naam', 'klant'], errors='ignore')

df_verkoop_av = pd.merge(df_verkoop_av, df_monteur_av, left_on='monteur', right_on='monteurnr', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['woonplaats', 'uurloon', 'filiaal', 'monteurnr'], errors='ignore')
df_verkoop_av = pd.merge(df_verkoop_av, df_monteur_combined[['naam','monteurnr']], left_on='naam', right_on='naam', how='left')
df_verkoop_av = df_verkoop_av.drop(columns=['naam', 'monteur'], errors='ignore')

# --- Fiets-verkopen ---
df_verkoop_fv = pd.merge(df_verkoop_fv, df_fiets_fv, left_on='fiets', right_on='fietsnr', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['fietsnr', 'soort', 'merk', 'standaardprijs', 'kleur', 'fabrikant', 'inkoopprijs'], errors='ignore')
df_verkoop_fv = pd.merge(df_verkoop_fv, df_product_combined[['type_product','productnr','standaardprijs']], left_on='type', right_on='type_product', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['type', 'soort', 'merk', 'kleur', 'fiets'], errors='ignore')

df_verkoop_fv = pd.merge(df_verkoop_fv, df_klant_fv, left_on='klant', right_on='klantnr', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['klantnr', 'geslacht', 'adres', 'woonplaats', 'geboortedatum'], errors='ignore')
df_verkoop_fv = pd.merge(df_verkoop_fv, df_klant_combined[['naam','klantnr']], left_on='naam', right_on='naam', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['naam', 'klant'], errors='ignore')

df_verkoop_fv = pd.merge(df_verkoop_fv, df_monteur_fv, left_on='monteur', right_on='monteurNr', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['woonplaats', 'uurloon', 'filiaal', 'monteurNr'], errors='ignore')
df_verkoop_fv = pd.merge(df_verkoop_fv, df_monteur_combined[['naam','monteurnr']], left_on='naam', right_on='naam', how='left')
df_verkoop_fv = df_verkoop_fv.drop(columns=['naam', 'monteur'], errors='ignore')

# --- Samenvoegen ---
df_verkoop_av = df_verkoop_av.drop('accessoire_verkoopnr', axis=1)
df_verkoop_fv = df_verkoop_fv.drop('fiets_verkoopnr', axis=1)

df_verkoop_combined = pd.concat([df_verkoop_av, df_verkoop_fv]).drop_duplicates().reset_index(drop=True)
df_verkoop_combined.insert(0, 'verkoopnr', range(1, len(df_verkoop_combined) + 1))
df_verkoop_combined['korting']     = df_verkoop_combined['standaardprijs'] - df_verkoop_combined['verkoopprijs']
df_verkoop_combined['totaalprijs'] = df_verkoop_combined['verkoopprijs'] * df_verkoop_combined['aantal']
df_verkoop_combined = df_verkoop_combined.drop(columns=['type_product', 'categorie', 'omzet'], errors='ignore')

# --- Temporele join: koppel surrogate keys op transactiedatum ---
# Zorg dat datum kolommen in string formaat zijn voor vergelijking
df_verkoop_combined['datum'] = df_verkoop_combined['datum'].astype(str)

# Temporele join voor klant
df_temp = df_verkoop_combined.merge(
    df_klant_scd2[['klantnr', 'klant_sk', 'geldig_van', 'geldig_tot']], 
    on='klantnr', 
    how='left'
)
df_temp = df_temp[
    (df_temp['datum'] >= df_temp['geldig_van']) & 
    (df_temp['datum'] <= df_temp['geldig_tot'])
].copy()
df_verkoop_combined = df_temp.drop(columns=['geldig_van', 'geldig_tot'])

# Temporele join voor monteur
df_temp = df_verkoop_combined.merge(
    df_monteur_scd2[['monteurnr', 'monteur_sk', 'geldig_van', 'geldig_tot']], 
    on='monteurnr', 
    how='left'
)
df_temp = df_temp[
    (df_temp['datum'] >= df_temp['geldig_van']) & 
    (df_temp['datum'] <= df_temp['geldig_tot'])
].copy()
df_verkoop_combined = df_temp.drop(columns=['geldig_van', 'geldig_tot'])

# Temporele join voor product
df_temp = df_verkoop_combined.merge(
    df_product_scd2[['productnr', 'product_sk', 'geldig_van', 'geldig_tot']], 
    on='productnr', 
    how='left'
)
df_temp = df_temp[
    (df_temp['datum'] >= df_temp['geldig_van']) & 
    (df_temp['datum'] <= df_temp['geldig_tot'])
].copy()
df_verkoop_combined = df_temp.drop(columns=['geldig_van', 'geldig_tot'])

# Verwijder eventuele merge-suffixen
df_verkoop_combined = df_verkoop_combined.rename(columns={'klant_sk_y': 'klant_sk'})
df_verkoop_combined = df_verkoop_combined.drop(
    columns=[c for c in df_verkoop_combined.columns if c.endswith('_x') or c.endswith('_y')],
    errors='ignore'
)

df_verkoop_combined.rename(columns={
    'monteurnr': 'monteur_sk',
    'productnr': 'product_sk',
    'klantnr': 'klant_sk'
}, inplace=True)

_leeg_tabel('Feit_Verkoop', DWH)
df_verkoop_combined.to_sql('Feit_Verkoop', DWH, if_exists='append', index=False)

df_verkoop_combined

,verkoopnr,datum,aantal,verkoopprijs,standaardprijs,product_sk,klant_sk,monteur_sk,korting,totaalprijs,klant_sk,monteur_sk,product_sk
0,1,2024-01-23,3,13.90,14.99,1,9,5,1.09,41.70,9,5,1
1,2,2024-10-03,2,11.27,14.99,1,7,7,3.72,22.54,7,7,1
2,3,2024-10-30,2,37.21,39.95,7,5,7,2.74,74.42,5,7,7
3,4,2024-11-16,1,10.71,11.99,10,9,2,1.28,10.71,9,2,10
4,5,2024-01-02,4,15.20,14.99,1,25,9,-0.21,60.80,25,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,286,2024-02-14,3,1159.80,962.33,44,6,8,-197.47,3479.40,6,8,44
286,287,2024-09-16,3,1075.78,980.16,111,24,7,-95.62,3227.34,24,7,111
287,288,2024-04-27,3,1581.13,341.93,81,9,3,-1239.20,4743.39,9,3,81
288,289,2024-11-21,2,911.00,547.79,46,8,2,-363.21,1822.00,8,2,46


## Feit_Inkoop

In [495]:
df_inkoop_ai = pd.read_sql_query('SELECT * FROM accessoire_inkoop_inkoop', SDM)
df_inkoop_fi = pd.read_sql_query('SELECT * FROM fiets_inkoop_inkoop', SDM)

df_accessoire_ai = pd.read_sql_query('SELECT * FROM accessoire_inkoop_accessoire', SDM)
df_fiets_fi      = pd.read_sql_query('SELECT * FROM fiets_inkoop_fiets', SDM)

df_inkoop_ai = pd.merge(df_inkoop_ai, df_accessoire_ai, left_on='accessoire', right_on='accessoirenr', how='left')
df_inkoop_ai = df_inkoop_ai.drop(columns=['accessoirenr', 'standaardprijs', 'soort', 'accessoire', 'inkoopprijs', 'leverancier'], errors='ignore')
df_inkoop_ai = pd.merge(df_inkoop_ai, df_product_combined[['type_product','productnr','inkoopprijs','partnernr']], left_on='naam', right_on='type_product', how='left')
df_inkoop_ai = df_inkoop_ai.drop(columns=['type', 'soort', 'standaardprijs', 'merk', 'kleur', 'naam', 'type_product'], errors='ignore')

df_inkoop_fi = pd.merge(df_inkoop_fi, df_fiets_fi, left_on='fiets', right_on='fietsnr', how='left')
df_inkoop_fi = df_inkoop_fi.drop(columns=['merk', 'fietsnr', 'soort', 'standaardprijs', 'inkoopprijs', 'kleur', 'fabrikant', 'fiets'], errors='ignore')
df_inkoop_fi = pd.merge(df_inkoop_fi, df_product_combined[['type_product','productnr','inkoopprijs','partnernr']], left_on='type', right_on='type_product', how='left')
df_inkoop_fi = df_inkoop_fi.drop(columns=['type', 'standaardprijs', 'soort', 'merk', 'kleur', 'type_product'], errors='ignore')

df_inkoop_ai = df_inkoop_ai.drop('inkoopnr', axis=1, errors='ignore')
df_inkoop_fi = df_inkoop_fi.drop('inkoopnr', axis=1, errors='ignore')

df_inkoop_combined = pd.concat([df_inkoop_ai, df_inkoop_fi]).drop_duplicates().reset_index(drop=True)
df_inkoop_combined.insert(0, 'inkoopnr', range(1, len(df_inkoop_combined) + 1))
df_inkoop_combined['totaalprijs'] = df_inkoop_combined['inkoopprijs'] * df_inkoop_combined['aantal']
df_inkoop_combined = df_inkoop_combined.drop(columns=['categorie', 'omzet'], errors='ignore')

# Feit_Inkoop heeft geen datum-kolom per regel maar wel inkoopmaand/jaar;
# we koppelen producten op basis van de huidige actieve versie (is_huidig=1)
df_product_huidig = df_product_scd2[df_product_scd2['is_huidig'] == 1][['productnr','product_sk']]
df_partner_huidig = df_partner_scd2[df_partner_scd2['is_huidig'] == 1][['partnernr','partner_sk']]

df_inkoop_combined = df_inkoop_combined.merge(df_product_huidig, on='productnr', how='left')
df_inkoop_combined = df_inkoop_combined.merge(df_partner_huidig, on='partnernr', how='left')

# Drop rows with missing surrogate keys before writing to database
df_inkoop_combined = df_inkoop_combined.dropna(subset=['product_sk', 'partner_sk'])

df_inkoop_combined.rename(columns={
    'productnr': 'product_sk',
    'partnernr': 'partner_sk'
}, inplace=True)

_leeg_tabel('Feit_Inkoop', DWH)
df_inkoop_combined.to_sql('Feit_Inkoop', DWH, if_exists='append', index=False)

df_inkoop_combined

,inkoopnr,inkoopmaand,inkoopjaar,aantal,product_sk,inkoopprijs,partner_sk,totaalprijs,product_sk,partner_sk
0,1,4,2024,38,7,24.90,2,946.20,7,2
1,2,8,2024,31,6,18.75,2,581.25,6,2
2,3,1,2024,41,2,7.25,1,297.25,2,1
3,4,1,2024,21,1,8.50,1,178.50,1,1
4,5,10,2024,31,4,11.40,1,353.40,4,1
...,...,...,...,...,...,...,...,...,...,...
153,154,4,2024,7,99,465.36,11,3257.52,99,11
154,155,12,2024,5,86,780.62,9,3903.10,86,9
155,156,11,2024,2,61,714.39,7,1428.78,61,7
156,157,7,2024,1,61,714.39,7,714.39,61,7


## Feit_Onderhoud

In [496]:
df_onderhoud_o = pd.read_sql_query('SELECT * FROM onderhoud_onderhoud', SDM)
df_fiets_o     = pd.read_sql_query('SELECT * FROM onderhoud_fiets', SDM)
df_monteur_o   = pd.read_sql_query('SELECT * FROM onderhoud_monteur', SDM)

df_onderhoud_o = pd.merge(df_onderhoud_o, df_fiets_o, left_on='fiets', right_on='fietsnr', how='left')
df_onderhoud_o = df_onderhoud_o.drop(columns=['fietsnr', 'soort', 'merk', 'standaardprijs', 'inkoopprijs', 'kleur', 'fabrikant'], errors='ignore')
df_onderhoud_o = pd.merge(df_onderhoud_o, df_product_combined[['type_product','productnr']], left_on='type', right_on='type_product', how='left')
df_onderhoud_o = df_onderhoud_o.drop(columns=['standaardprijs', 'soort', 'merk', 'kleur', 'partnernr', 'inkoopprijs', 'fiets'], errors='ignore')

df_onderhoud_o = pd.merge(df_onderhoud_o, df_monteur_o, left_on='monteur', right_on='monteurNr', how='left')
df_onderhoud_o = df_onderhoud_o.drop(columns=['monteurNr', 'woonplaats', 'uurloon', 'filiaal'], errors='ignore')
df_onderhoud_o = pd.merge(df_onderhoud_o, df_monteur_combined[['naam','monteurnr']], left_on='naam', right_on='naam', how='left')
df_onderhoud_o = df_onderhoud_o.drop(columns=['naam', 'woonplaats', 'uurloon', 'filiaal', 'monteur'], errors='ignore')

df_onderhoud_o['duur'] = (
    pd.to_datetime(df_onderhoud_o['eindtijd'].str[:-1], format='%H:%M:%S.%f') -
    pd.to_datetime(df_onderhoud_o['starttijd'].str[:-1], format='%H:%M:%S.%f')
)
df_onderhoud_o['duur'] = df_onderhoud_o['duur'].apply(
    lambda x: f"{int(x.total_seconds()//3600):02}:{int((x.total_seconds()%3600)//60):02}"
)

df_onderhoud_o = df_onderhoud_o.drop('onderhoudnr', axis=1)
df_onderhoud_o.insert(0, 'onderhoudnr', range(1, len(df_onderhoud_o) + 1))
df_onderhoud_o = df_onderhoud_o.drop(columns=['type', 'type_product', 'categorie', 'omzet'], errors='ignore')

# Temporele join op onderhoudsdatum
df_onderhoud_o = haal_sk_op(
    df_feit      = df_onderhoud_o,
    dim_df       = df_monteur_scd2,
    business_key = 'monteurnr',
    datum_kolom  = 'datum',
    sk_kolom     = 'monteur_sk'
)
df_product_huidig = df_product_scd2[df_product_scd2['is_huidig'] == 1][['productnr','product_sk']]
df_onderhoud_o = df_onderhoud_o.merge(df_product_huidig, on='productnr', how='left')
df_onderhoud_o.rename(columns={'monteurnr': 'monteur_sk', 'productnr': 'product_sk'}, inplace=True)

_leeg_tabel('Feit_Onderhoud', DWH)
df_onderhoud_o.to_sql('Feit_Onderhoud', DWH, if_exists='append', index=False)
DWH.commit()
df_onderhoud_o

,onderhoudnr,datum,starttijd,eindtijd,product_sk,monteur_sk,duur,monteur_sk,product_sk
0,1,2024-12-28,15:00:00.0000000,16:00:00.0000000,23,15,01:00,15,23
1,2,2024-02-05,14:00:00.0000000,15:00:00.0000000,19,8,01:00,8,19
2,3,2024-08-01,10:00:00.0000000,11:00:00.0000000,15,15,01:00,15,15
3,4,2024-11-16,10:00:00.0000000,11:00:00.0000000,17,2,01:00,2,17
4,5,2024-08-01,11:00:00.0000000,12:00:00.0000000,36,16,01:00,16,36
5,6,2024-12-08,13:00:00.0000000,14:00:00.0000000,19,14,01:00,14,19
6,7,2024-05-14,11:00:00.0000000,12:00:00.0000000,16,8,01:00,8,16
7,8,2024-10-10,09:00:00.0000000,10:00:00.0000000,41,3,01:00,3,41
8,9,2024-06-29,08:00:00.0000000,09:00:00.0000000,19,13,01:00,13,19
9,10,2024-08-21,08:00:00.0000000,09:00:00.0000000,28,2,01:00,2,28


## Close connections

In [497]:
SDM.close()
DWH.close()